# ALDIMI-PREDICT | Logistica — Limpieza y Modelado v2
**Modelos:** XGBoost + Random Forest (reemplaza Ridge)
**Evaluacion:** Validacion cruzada + analisis de errores
**Integracion:** GCP BigQuery Bronze / Silver / Gold
**Proyecto GCP:** 413462127752 | mlaldimi

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost google-cloud-bigquery -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json, os, zipfile, joblib, warnings
warnings.filterwarnings("ignore")
from datetime import datetime
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb

GCP_PROJECT    = "mlaldimi"
GCP_PROJECT_ID = "413462127752"
DATASET_ID     = "mlaldimi"
TABLE_BRONZE   = f"{GCP_PROJECT}.{DATASET_ID}.bronze_logistica"
TABLE_SILVER   = f"{GCP_PROJECT}.{DATASET_ID}.silver_logistica"
TABLE_GOLD     = f"{GCP_PROJECT}.{DATASET_ID}.gold_logistica"
SEED = 42
np.random.seed(SEED)
print(f"GCP: {GCP_PROJECT_ID} | Dataset: {DATASET_ID}")

## 1. Carga de Datos — GCP Gold / local / sintetico

In [ ]:
def load_from_gcp_gold():
    try:
        from google.cloud import bigquery
        client = bigquery.Client(project=GCP_PROJECT)
        df = client.query(f"SELECT * FROM ").to_dataframe()
        print(f"Gold GCP cargado: {len(df):,} filas")
        return df, "gold_gcp"
    except Exception as e:
        print(f"GCP Gold no disponible: {e}")
        return None, None

def load_local():
    files = {"train": "train.csv", "stores": "stores.csv", "items": "items.csv",
             "oil": "oil.csv", "holidays": "holidays_events.csv", "transactions": "transactions.csv"}
    if not all(os.path.exists(v) for v in files.values()):
        return None, None
    train = pd.read_csv(files["train"], parse_dates=["date"], dtype={"onpromotion": object}, nrows=5_000_000)
    stores = pd.read_csv(files["stores"])
    items = pd.read_csv(files["items"])
    oil = pd.read_csv(files["oil"], parse_dates=["date"])
    hol = pd.read_csv(files["holidays"], parse_dates=["date"])
    trans = pd.read_csv(files["transactions"], parse_dates=["date"])
    PRIORIDAD = {"Holiday":0,"Event":1,"Additional":2,"Bridge":3,"Transfer":4,"Work Day":5}
    hol_nac = (hol[hol["locale"]=="National"]
               .assign(prio=lambda x: x["type"].map(PRIORIDAD)).sort_values("prio")
               .drop_duplicates("date", keep="first")[["date","type","transferred"]]
               .rename(columns={"type":"holiday_type"}))
    df = (train.merge(items[["item_nbr","family","class","perishable"]], on="item_nbr", how="left")
          .merge(stores[["store_nbr","city","state","type","cluster"]], on="store_nbr", how="left")
          .merge(oil, on="date", how="left")
          .merge(hol_nac, on="date", how="left")
          .merge(trans.rename(columns={"transactions":"n_transactions"}), on=["date","store_nbr"], how="left"))
    FAMILIAS = ["PRODUCE","MEATS","SEAFOOD","DAIRY","BREAD/BAKERY","EGGS","POULTRY",
                "BEVERAGES","GROCERY I","GROCERY II","DELI","PREPARED FOODS"]
    df = df[df["family"].isin(FAMILIAS)].copy()
    print(f"Local cargado: {len(df):,} filas")
    return df, "local"

def make_synthetic(n=50000, seed=42):
    rng = np.random.default_rng(seed)
    families = ["PRODUCE","MEATS","DAIRY","BEVERAGES","GROCERY I","BREAD/BAKERY","EGGS","POULTRY"]
    cities   = ["Quito","Guayaquil","Cuenca","Ambato","Manta"]
    dates    = pd.date_range("2013-01-01", periods=365*4, freq="D")
    return pd.DataFrame({
        "date": pd.to_datetime(rng.choice(dates, n)),
        "store_nbr": rng.integers(1, 55, n).astype(np.int32),
        "item_nbr": rng.integers(1, 4100, n).astype(np.int32),
        "family": rng.choice(families, n),
        "unit_sales": np.abs(rng.normal(8, 5, n)).astype(np.float32),
        "onpromotion": rng.choice(["False","True"], n, p=[0.8, 0.2]),
        "perishable": rng.choice([0,1], n, p=[0.45, 0.55]),
        "city": rng.choice(cities, n),
        "state": "Pichincha",
        "type": rng.choice(["A","B","C","D","E"], n),
        "cluster": rng.integers(1, 18, n),
        "dcoilwtico": rng.uniform(25, 100, n),
        "holiday_type": rng.choice(["Normal","Holiday","Event"], n, p=[0.85,0.10,0.05]),
        "transferred": rng.choice([False,True], n, p=[0.95,0.05]),
        "n_transactions": rng.integers(100, 5000, n).astype(float),
    })

df_raw, fuente = load_from_gcp_gold()
if df_raw is None:
    df_raw, fuente = load_local()
if df_raw is None:
    df_raw = make_synthetic()
    fuente = "synthetic_demo"
    print("Usando dataset sintetico (50k registros)")
print(f"Fuente: {fuente} | Shape: {df_raw.shape}")
df_raw.head(3)

## 2. Limpieza

In [ ]:
df = df_raw.copy()
df["unit_sales"] = np.maximum(pd.to_numeric(df["unit_sales"], errors="coerce").fillna(0), 0)
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill() if "dcoilwtico" in df.columns else 52.0
df["holiday_type"] = df["holiday_type"].fillna("Normal")
df["transferred"] = df["transferred"].fillna(False)
df["onpromotion"] = df["onpromotion"].fillna(False).astype(str).str.lower().isin(["true","1"]).astype(int)
if "n_transactions" in df.columns:
    df["n_transactions"] = df["n_transactions"].fillna(df["n_transactions"].median())
else:
    df["n_transactions"] = 500.0
# Winsorization
q99 = df["unit_sales"].quantile(0.99)
df["unit_sales"] = df["unit_sales"].clip(0, q99)
q99_t = df["n_transactions"].quantile(0.99)
df["n_transactions"] = df["n_transactions"].clip(0, q99_t)
print(f"Limpieza ok. Nulos restantes: {df.isnull().sum().sum()}")

# Guardar Silver en GCP
def upload_silver(df):
    try:
        from google.cloud import bigquery
        client = bigquery.Client(project=GCP_PROJECT)
        job = client.load_table_from_dataframe(df.head(10000), TABLE_SILVER,
              job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
        job.result()
        print(f"Silver GCP actualizado: {TABLE_SILVER}")
    except Exception as e:
        print(f"Silver GCP (local mode): {e}")
upload_silver(df)

## 3. Feature Engineering

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["dia_semana"]  = df["date"].dt.dayofweek
df["es_finde"]    = (df["dia_semana"] >= 5).astype(int)
df["mes"]         = df["date"].dt.month
df["semana_anio"] = df["date"].dt.isocalendar().week.astype(int)
df["anio"]        = df["date"].dt.year
df["trimestre"]   = df["date"].dt.quarter
df["es_festivo"]  = (df["holiday_type"] != "Normal").astype(int)
df["log_unit_sales"] = np.log1p(df["unit_sales"])

col_type = "type" if "type" in df.columns else "store_type"
df["store_type_enc"] = df[col_type].map({"A":1,"B":2,"C":3,"D":4,"E":5}).fillna(3).astype(int)
df["family_enc"] = df["family"].astype("category").cat.codes
df["city_enc"]   = df["city"].astype("category").cat.codes if "city" in df.columns else 0
if "cluster" not in df.columns:
    df["cluster"] = 5

mm = MinMaxScaler()
df[["dcoilwtico_scaled","n_transactions_scaled"]] = mm.fit_transform(df[["dcoilwtico","n_transactions"]])

# Lag features
print("Creando lag features...")
group_col = ["store_nbr","item_nbr"] if "item_nbr" in df.columns else ["store_nbr","family"]
df = df.sort_values(group_col + ["date"])
grp = df.groupby(group_col)["unit_sales"]
df["lag_1"]    = grp.shift(1).fillna(0)
df["lag_7"]    = grp.shift(7).fillna(0)
df["lag_14"]   = grp.shift(14).fillna(0)
df["media_7d"] = grp.transform(lambda x: x.shift(1).rolling(7, min_periods=3).mean()).fillna(0)
df["media_14d"]= grp.transform(lambda x: x.shift(1).rolling(14,min_periods=5).mean()).fillna(0)
df["std_7d"]   = grp.transform(lambda x: x.shift(1).rolling(7, min_periods=3).std()).fillna(0)

# Targets
print("Construyendo targets demand7 y demand14...")
grp2 = df.groupby(group_col)["unit_sales"]
df["demand7"]  = grp2.transform(lambda x: x.shift(-1).rolling(7,  min_periods=4).sum().shift(-6)).fillna(0)
df["demand14"] = grp2.transform(lambda x: x.shift(-1).rolling(14, min_periods=8).sum().shift(-13)).fillna(0)
if "perishable" not in df.columns:
    perece_fam = {"PRODUCE","DAIRY","MEATS","SEAFOOD","POULTRY","EGGS","DELI","BREAD/BAKERY","PREPARED FOODS"}
    df["perishable"] = df["family"].isin(perece_fam).astype(int)
df["perece"] = df["perishable"].fillna(0).astype(int)
print(f"Feature engineering ok. Shape: {df.shape}")

## 4. Preparacion Train/Test + Guardar Gold

In [ ]:
FEATURES = [
    "lag_1","lag_7","lag_14","media_7d","media_14d","std_7d",
    "log_unit_sales","onpromotion",
    "dcoilwtico_scaled","n_transactions_scaled",
    "es_festivo","dia_semana","es_finde","mes","semana_anio","anio","trimestre",
    "store_type_enc","family_enc","city_enc","cluster"
]
FEATURES_OK = [f for f in FEATURES if f in df.columns]
print(f"Features: {len(FEATURES_OK)}/{len(FEATURES)}")

df_model = df.dropna(subset=["demand7","demand14"]).query("demand7>=0 and demand14>=0")
df_model = df_model.sort_values("date")
corte    = int(len(df_model) * 0.80)
train_df = df_model.iloc[:corte]
test_df  = df_model.iloc[corte:]

X_train  = train_df[FEATURES_OK].fillna(0)
X_test   = test_df[FEATURES_OK].fillna(0)
y7_train = train_df["demand7"]
y7_test  = test_df["demand7"]
y14_train= train_df["demand14"]
y14_test = test_df["demand14"]
yp_train = train_df["perece"]
yp_test  = test_df["perece"]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Guardar Gold en GCP
def upload_gold(df, features):
    try:
        from google.cloud import bigquery
        client = bigquery.Client(project=GCP_PROJECT)
        cols   = [c for c in features + ["demand7","demand14","perece"] if c in df.columns]
        df_g   = df[cols].head(50000).copy()
        df_g["ingestion_ts"] = pd.Timestamp.utcnow().isoformat()
        job = client.load_table_from_dataframe(df_g, TABLE_GOLD,
              job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
        job.result()
        print(f"Gold GCP: {TABLE_GOLD} ({len(df_g):,} filas)")
    except Exception as e:
        print(f"Gold GCP (local): {e}")
upload_gold(df_model, FEATURES_OK)

## 5. Modelado — XGBoost y Random Forest
Se eliminó Ridge Regression. Se usan algoritmos avanzados con ajuste de hiperparámetros.

In [ ]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100

def eval_reg(name, y_true, y_pred):
    w   = wape(np.asarray(y_true), np.asarray(y_pred))
    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse= np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:22s} | WAPE%={w:.2f}  R2={r2:.4f}  MAE={mae:.3f}  RMSE={rmse:.3f}")
    return {"Modelo":name,"WAPE%":round(w,2),"R2":round(r2,4),"MAE":round(mae,3),"RMSE":round(rmse,3)}

# GridSearchCV ampliado — hiperparametros ajustados segun metadata optimo
sample_idx = np.random.choice(len(X_train), min(8000,len(X_train)), replace=False)
X_gs = X_train.iloc[sample_idx]; y7_gs = y7_train.iloc[sample_idx]

# XGBoost: incluye regularizacion L1/L2 y min_child_weight
xgb_grid = {
    "n_estimators":     [200, 400],
    "max_depth":        [4, 6],
    "learning_rate":    [0.05, 0.1],
    "subsample":        [0.8],
    "colsample_bytree": [0.8],
    "min_child_weight": [3, 5],
    "reg_alpha":        [0.0, 0.1],
    "reg_lambda":       [1.0],
}
gs_xgb = GridSearchCV(
    XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=-1, verbosity=0),
    xgb_grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=1
)
gs_xgb.fit(X_gs, y7_gs)
print(f"XGBoost best params demand7: {gs_xgb.best_params_}")
print(f"XGBoost best MAE CV:          {-gs_xgb.best_score_:.3f}")

# Random Forest: incluye max_features para controlar varianza
rf_grid = {
    "n_estimators":    [100, 200],
    "max_depth":       [8, 12, None],
    "min_samples_leaf":[3, 5],
    "max_features":    ["sqrt", 0.5],
}
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=-1),
    rf_grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=1
)
gs_rf.fit(X_gs, y7_gs)
print(f"RF best params: {gs_rf.best_params_}")
print(f"RF best MAE CV: {-gs_rf.best_score_:.3f}")

In [ ]:
print("="*60)
print("DEMAND 7")

xgb7 = XGBRegressor(**gs_xgb.best_params_, objective="reg:squarederror", random_state=SEED, n_jobs=-1, verbosity=0)
xgb7.fit(X_train, y7_train, eval_set=[(X_test,y7_test)], verbose=False)
res_xgb7 = eval_reg("XGBoost d7", y7_test, xgb7.predict(X_test))

rf7 = RandomForestRegressor(**gs_rf.best_params_, random_state=SEED, n_jobs=-1)
rf7.fit(X_train, y7_train)
res_rf7 = eval_reg("Random Forest d7", y7_test, rf7.predict(X_test))

lgbm7 = lgb.LGBMRegressor(n_estimators=400,max_depth=6,learning_rate=0.05,subsample=0.8,
                            colsample_bytree=0.8,random_state=SEED,n_jobs=-1,verbose=-1)
lgbm7.fit(X_train, y7_train)
res_lgbm7 = eval_reg("LightGBM d7", y7_test, lgbm7.predict(X_test))

print("="*60)
print("DEMAND 14")

y14_gs = y14_train.iloc[sample_idx]
gs_xgb14 = GridSearchCV(XGBRegressor(objective="reg:squarederror",random_state=SEED,n_jobs=-1,verbosity=0),
                          xgb_grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=0)
gs_xgb14.fit(X_gs, y14_gs)

xgb14 = XGBRegressor(**gs_xgb14.best_params_, objective="reg:squarederror", random_state=SEED, n_jobs=-1, verbosity=0)
xgb14.fit(X_train, y14_train, eval_set=[(X_test,y14_test)], verbose=False)
res_xgb14 = eval_reg("XGBoost d14", y14_test, xgb14.predict(X_test))

rf14 = RandomForestRegressor(**gs_rf.best_params_, random_state=SEED, n_jobs=-1)
rf14.fit(X_train, y14_train)
res_rf14 = eval_reg("Random Forest d14", y14_test, rf14.predict(X_test))

lgbm14 = lgb.LGBMRegressor(n_estimators=400,max_depth=6,learning_rate=0.05,subsample=0.8,
                             colsample_bytree=0.8,random_state=SEED,n_jobs=-1,verbose=-1)
lgbm14.fit(X_train, y14_train)
res_lgbm14 = eval_reg("LightGBM d14", y14_test, lgbm14.predict(X_test))

print("
--- Tabla comparativa ---")
print(pd.DataFrame([res_xgb7,res_rf7,res_lgbm7,res_xgb14,res_rf14,res_lgbm14]).to_string(index=False))

## 6. Validacion Cruzada — Evaluacion Robusta

In [ ]:
print("="*60)
print("VALIDACION CRUZADA K=5 — demand7")
kf = KFold(n_splits=5, shuffle=False)
for name, model_cv in [("XGBoost", XGBRegressor(**gs_xgb.best_params_,objective="reg:squarederror",random_state=SEED,n_jobs=-1,verbosity=0)),
                        ("RandomForest", RandomForestRegressor(**gs_rf.best_params_,random_state=SEED,n_jobs=-1))]:
    cv = cross_validate(model_cv, X_train, y7_train, cv=kf,
                         scoring=["neg_mean_absolute_error","r2"], return_train_score=True, n_jobs=-1)
    print(f"
{name}:")
    print(f"  Test MAE  = {-cv["test_neg_mean_absolute_error"].mean():.3f} +/- {cv["test_neg_mean_absolute_error"].std():.3f}")
    print(f"  Test R2   = {cv["test_r2"].mean():.4f} +/- {cv["test_r2"].std():.4f}")
    gap = cv["train_r2"].mean() - cv["test_r2"].mean()
    print(f"  Overfitting R2 gap = {gap:.4f} {"(alto!)" if gap>0.05 else "(OK)"}")

## 7. Clasificacion de Perecibilidad

In [ ]:
print(f"Perecibles train: {yp_train.mean():.1%} | test: {yp_test.mean():.1%}")

xgb_perece = XGBClassifier(n_estimators=200,max_depth=6,learning_rate=0.1,
                             use_label_encoder=False,eval_metric="logloss",
                             random_state=SEED,n_jobs=-1,verbosity=0)
xgb_perece.fit(X_train, yp_train)
pred_xp = xgb_perece.predict(X_test)
prob_xp = xgb_perece.predict_proba(X_test)[:,1]
print(f"XGBoost Perece: Acc={accuracy_score(yp_test,pred_xp):.4f} AUC={roc_auc_score(yp_test,prob_xp):.4f}")
print(classification_report(yp_test, pred_xp, target_names=["No Perece","Perece"]))

rf_perece = RandomForestClassifier(n_estimators=200,max_depth=12,random_state=SEED,n_jobs=-1)
rf_perece.fit(X_train, yp_train)
pred_rfp = rf_perece.predict(X_test)
prob_rfp = rf_perece.predict_proba(X_test)[:,1]
print(f"RF Perece:      Acc={accuracy_score(yp_test,pred_rfp):.4f} AUC={roc_auc_score(yp_test,prob_rfp):.4f}")

## 8. Analisis de Errores

In [ ]:
pred_xgb7_  = xgb7.predict(X_test)
pred_rf7_   = rf7.predict(X_test)
pred_xgb14_ = xgb14.predict(X_test)

fig, axes = plt.subplots(2, 3, figsize=(18,10))

res_xgb7v  = y7_test.values  - pred_xgb7_
axes[0,0].hist(res_xgb7v, bins=50, color="#0f766e", alpha=0.8, edgecolor="white")
axes[0,0].axvline(0, color="red", ls="--", lw=2)
axes[0,0].set_title("Residuos — XGBoost demand7", fontweight="bold")

s = min(2000, len(y7_test))
axes[0,1].scatter(y7_test[:s], pred_xgb7_[:s], alpha=0.3, color="#0f766e", s=8)
lim = max(y7_test[:s].max(), pred_xgb7_[:s].max())
axes[0,1].plot([0,lim],[0,lim],"r--",lw=2)
axes[0,1].set_title("Real vs Predicho — XGBoost d7", fontweight="bold")

fi = pd.Series(xgb7.feature_importances_, index=FEATURES_OK).sort_values().tail(15)
axes[0,2].barh(fi.index, fi.values, color="#0f766e", alpha=0.85)
axes[0,2].set_title("Importancia Features — XGBoost d7", fontweight="bold")

res_rf7v = y7_test.values - pred_rf7_
axes[1,0].hist(res_rf7v, bins=50, color="#2563eb", alpha=0.8, edgecolor="white")
axes[1,0].axvline(0, color="red", ls="--", lw=2)
axes[1,0].set_title("Residuos — Random Forest demand7", fontweight="bold")

names_bar = ["XGB d7","RF d7","LGBM d7","XGB d14","RF d14","LGBM d14"]
wapes_bar = [res_xgb7["WAPE%"],res_rf7["WAPE%"],res_lgbm7["WAPE%"],
             res_xgb14["WAPE%"],res_rf14["WAPE%"],res_lgbm14["WAPE%"]]
bars = axes[1,1].bar(names_bar, wapes_bar, color=["#0f766e"]*3+["#2563eb"]*3, alpha=0.85, edgecolor="white")
axes[1,1].axhline(20, color="orange", ls=":", lw=2, label="Objetivo 20%")
axes[1,1].set_title("WAPE% por Modelo", fontweight="bold")
axes[1,1].legend()
for bar, val in zip(bars, wapes_bar):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, val+0.1, f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")

cm_p = confusion_matrix(yp_test, pred_xp)
sns.heatmap(cm_p, annot=True, fmt="d", cmap="Greens", ax=axes[1,2],
            xticklabels=["No Perece","Perece"], yticklabels=["No Perece","Perece"],
            cbar=False, annot_kws={"size":13,"weight":"bold"})
axes[1,2].set_title("Confusion Matrix — XGBoost Perece", fontweight="bold")

plt.suptitle("Analisis de Errores — ALDIMI-PREDICT Logistica", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("analisis_errores_logistica.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Serializacion y Metadata

In [ ]:
out_dir = "../../../Dashboard/models/favorita_modelos"
os.makedirs(out_dir, exist_ok=True)

joblib.dump(xgb7,       f"{out_dir}/xgb_demand7.pkl")
joblib.dump(xgb14,      f"{out_dir}/xgb_demand14.pkl")
joblib.dump(rf7,        f"{out_dir}/rf_demand7.pkl")
joblib.dump(rf14,       f"{out_dir}/rf_demand14.pkl")
joblib.dump(xgb_perece, f"{out_dir}/xgb_perece.pkl")
print("Modelos guardados:", os.listdir(out_dir))

meta = {
    "generado": datetime.now().isoformat(), "fuente": fuente,
    "gcp_project": GCP_PROJECT_ID, "gcp_dataset": DATASET_ID, "tabla_gold": TABLE_GOLD,
    "features": FEATURES_OK, "n_train": len(X_train), "n_test": len(X_test),
    "mejor_reg_d7":  "XGBoost" if res_xgb7["WAPE%"]  <= res_rf7["WAPE%"]  else "RandomForest",
    "mejor_reg_d14": "XGBoost" if res_xgb14["WAPE%"] <= res_rf14["WAPE%"] else "RandomForest",
    "mejor_cls":     "XGBoost",
    "hiperparametros": {
        "xgb_d7":  gs_xgb.best_params_,
        "xgb_d14": gs_xgb14.best_params_,
        "rf_d7":   gs_rf.best_params_,
    },
    "kpi_targets": {
        "WAPE_demand7_pct": 20.0,
        "WAPE_demand14_pct": 20.0,
        "R2_demand7": 0.91,
        "R2_demand14": 0.91,
        "Acc_perece": 0.95,
        "AUC_perece": 0.95,
    },
    "resultados": {"demand7":[res_xgb7,res_rf7,res_lgbm7],"demand14":[res_xgb14,res_rf14,res_lgbm14],
                   "perece":{"XGBoost":{"Acc":round(accuracy_score(yp_test,pred_xp),4),
                                         "AUC":round(roc_auc_score(yp_test,prob_xp),4)},
                             "RF":     {"Acc":round(accuracy_score(yp_test,pred_rfp),4),
                                         "AUC":round(roc_auc_score(yp_test,prob_rfp),4)}}},
}
meta_path = "../../../Models/models/logistica/modelo_metadata.json"
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
with open(meta_path,"w",encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)
print(f"Metadata: {meta_path}")
print(json.dumps({k:v for k,v in meta.items() if k not in ["features","resultados","hiperparametros"]},indent=2,default=str))